# 20: LoRA Theory - Low-Rank Adaptation

This notebook covers the theoretical foundations of LoRA (Low-Rank Adaptation), one of the most important breakthroughs in parameter-efficient fine-tuning.

## Historical Context: The GPT-3 Era Problem (2020-2021)

### The Crisis
- **June 2020**: GPT-3 released with 175B parameters
- Full fine-tuning requires storing:
  - Model weights: 175B × 2 bytes (fp16) = 350GB
  - Optimizer states (Adam): 175B × 12 bytes = 2.1TB
  - **Total: ~2.5TB per fine-tuned model**
- Cost: $10K-100K per fine-tuning job
- Problem: Can't deploy 100 different fine-tuned models

### Previous Attempts
1. **Adapter Layers** (2019): Add small bottleneck layers
   - Problem: Adds inference latency
2. **Prompt Tuning** (2021): Optimize soft prompts
   - Problem: Quality gap vs full fine-tuning
3. **BitFit** (2021): Only tune bias terms
   - Problem: Limited expressiveness

### The LoRA Breakthrough (June 2021)
- Paper: "LoRA: Low-Rank Adaptation of Large Language Models" (Hu et al., Microsoft)
- Key insight: **Weight updates have low intrinsic dimensionality**
- Result: Match full fine-tuning quality with 0.01% parameters

## Matrix Decomposition Fundamentals

### Singular Value Decomposition (SVD)
Any matrix $W \in \mathbb{R}^{d \times k}$ can be decomposed:

$$W = U \Sigma V^T$$

where:
- $U \in \mathbb{R}^{d \times d}$: Left singular vectors
- $\Sigma \in \mathbb{R}^{d \times k}$: Singular values (diagonal)
- $V \in \mathbb{R}^{k \times k}$: Right singular vectors

### Low-Rank Approximation
Keep only top $r$ singular values:

$$W \approx W_r = U_r \Sigma_r V_r^T$$

where $r \ll \min(d, k)$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.linalg import svd

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 4)

In [ ]:
# Demonstrate low-rank approximation
def low_rank_approximation(W, rank):
    """
    Approximate matrix W with rank-r decomposition.
    
    Args:
        W: Original matrix (d, k)
        rank: Target rank r
    
    Returns:
        W_approx: Low-rank approximation
        U, S, Vt: SVD components
    """
    U, S, Vt = svd(W, full_matrices=False)
    
    # Keep only top-r components
    U_r = U[:, :rank]
    S_r = S[:rank]
    Vt_r = Vt[:rank, :]
    
    # Reconstruct
    W_approx = U_r @ np.diag(S_r) @ Vt_r
    
    return W_approx, U_r, S_r, Vt_r

# Example: 1000x1000 matrix
d, k = 1000, 1000
np.random.seed(42)

# Create a low-rank matrix + noise (simulating learned weights)
rank_true = 50
A = np.random.randn(d, rank_true)
B = np.random.randn(rank_true, k)
W = A @ B + 0.1 * np.random.randn(d, k)  # Low-rank + small noise

# Compute SVD
U, S, Vt = svd(W, full_matrices=False)

# Plot singular values
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Singular value spectrum
axes[0].semilogy(S, 'o-', markersize=2)
axes[0].axvline(rank_true, color='r', linestyle='--', label=f'True rank={rank_true}')
axes[0].set_xlabel('Index')
axes[0].set_ylabel('Singular Value')
axes[0].set_title('Singular Value Spectrum')
axes[0].legend()
axes[0].grid(True)

# Cumulative energy
energy = np.cumsum(S**2) / np.sum(S**2)
axes[1].plot(energy)
axes[1].axhline(0.90, color='r', linestyle='--', label='90% energy')
axes[1].axhline(0.99, color='g', linestyle='--', label='99% energy')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Energy')
axes[1].set_title('Energy Captured by Top-k Components')
axes[1].legend()
axes[1].grid(True)

# Approximation error vs rank
ranks = [1, 2, 4, 8, 16, 32, 64, 128, 256]
errors = []
for r in ranks:
    W_approx, _, _, _ = low_rank_approximation(W, r)
    error = np.linalg.norm(W - W_approx, 'fro') / np.linalg.norm(W, 'fro')
    errors.append(error)

axes[2].semilogy(ranks, errors, 'o-')
axes[2].axvline(rank_true, color='r', linestyle='--', label=f'True rank={rank_true}')
axes[2].set_xlabel('Rank r')
axes[2].set_ylabel('Relative Frobenius Error')
axes[2].set_title('Approximation Error vs Rank')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

print(f"Matrix shape: {W.shape}")
print(f"Rank for 90% energy: {np.argmax(energy > 0.90)}")
print(f"Rank for 99% energy: {np.argmax(energy > 0.99)}")

## The Intrinsic Dimensionality Hypothesis

### Key Observation
Li et al. (2018) discovered that fine-tuning operates in a **low intrinsic dimension**:
- Fine-tuned weights can be represented in a much smaller subspace
- Weight updates $\Delta W$ during fine-tuning are approximately low-rank
- This holds across different tasks and model sizes

### Empirical Evidence
From Aghajanyan et al. (2020):
- RoBERTa-base (125M params) can be fine-tuned in a 1000-dimensional subspace
- That's 0.0008% of the original parameter space
- Performance: 90% of full fine-tuning quality

### Why This Matters for LoRA
If $\Delta W$ is low-rank, we can represent it as:

$$\Delta W = BA$$

where:
- $B \in \mathbb{R}^{d \times r}$
- $A \in \mathbb{R}^{r \times k}$
- $r \ll \min(d, k)$

Parameters: $d \times r + r \times k$ instead of $d \times k$

In [ ]:
# Simulate intrinsic dimensionality
def simulate_fine_tuning_update(d, k, intrinsic_dim):
    """
    Simulate a weight update during fine-tuning with low intrinsic dimension.
    """
    # Generate random low-rank update
    U = np.random.randn(d, intrinsic_dim) / np.sqrt(intrinsic_dim)
    V = np.random.randn(intrinsic_dim, k) / np.sqrt(intrinsic_dim)
    delta_W = U @ V
    
    return delta_W

# Example: Typical transformer layer dimensions
d, k = 4096, 4096  # GPT-2 Large dimensions
intrinsic_dims = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]

# Compare parameter counts
full_params = d * k
lora_params = [(d * r + r * k) for r in intrinsic_dims]
param_ratios = [lp / full_params * 100 for lp in lora_params]

# Visualize parameter reduction
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute parameter counts
axes[0].bar(range(len(intrinsic_dims)), [full_params] * len(intrinsic_dims), 
            alpha=0.3, label='Full Fine-tuning', color='red')
axes[0].bar(range(len(intrinsic_dims)), lora_params, 
            alpha=0.8, label='LoRA', color='blue')
axes[0].set_xticks(range(len(intrinsic_dims)))
axes[0].set_xticklabels([f'r={r}' for r in intrinsic_dims], rotation=45)
axes[0].set_ylabel('Number of Parameters')
axes[0].set_title(f'Parameter Count: Full vs LoRA\n(Layer size: {d}x{k})')
axes[0].legend()
axes[0].grid(True, axis='y')
axes[0].set_yscale('log')

# Percentage of full parameters
axes[1].plot(intrinsic_dims, param_ratios, 'o-', linewidth=2, markersize=8)
axes[1].axhline(1.0, color='r', linestyle='--', label='1% of full params')
axes[1].axhline(0.1, color='g', linestyle='--', label='0.1% of full params')
axes[1].set_xlabel('LoRA Rank (r)')
axes[1].set_ylabel('% of Full Parameters')
axes[1].set_title('LoRA Parameters as % of Full Fine-tuning')
axes[1].legend()
axes[1].grid(True)
axes[1].set_xscale('log')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

# Print specific examples
print("Parameter Comparison:")
print(f"Full fine-tuning: {full_params:,} parameters")
print("\nLoRA with different ranks:")
for r, lp, ratio in zip(intrinsic_dims, lora_params, param_ratios):
    print(f"  r={r:3d}: {lp:,} params ({ratio:.4f}% of full)")

## LoRA Architecture

### Core Idea
For a pre-trained weight matrix $W_0 \in \mathbb{R}^{d \times k}$:

1. **Freeze** $W_0$ (no gradients, no updates)
2. Add a **trainable low-rank update**:

$$h = W_0 x + \Delta W x = W_0 x + BAx$$

where:
- $A \in \mathbb{R}^{r \times k}$: Randomly initialized (Gaussian)
- $B \in \mathbb{R}^{d \times r}$: Initialized to zero
- Initial behavior: $BA = 0$, so model starts from pre-trained state

### Scaling Factor
In practice, use:

$$h = W_0 x + \frac{\alpha}{r} BAx$$

where $\alpha$ is a constant (typically $\alpha = r$ or $\alpha = 2r$)
- Acts as learning rate multiplier for LoRA parameters
- Allows tuning LoRA strength without retraining

### Inference Optimization
After training, merge the weights:

$$W = W_0 + \frac{\alpha}{r} BA$$

- **Zero additional inference latency**
- Can swap between tasks by swapping $BA$ matrices
- Base model $W_0$ shared across all tasks

In [ ]:
import torch
import torch.nn as nn

class LoRALayer(nn.Module):
    """
    Simple LoRA layer implementation.
    
    Replaces: h = W @ x
    With: h = W @ x + (B @ A) @ x * (alpha / r)
    """
    def __init__(self, base_layer, r=8, alpha=16, dropout=0.0):
        """
        Args:
            base_layer: Original nn.Linear layer
            r: LoRA rank
            alpha: Scaling factor
            dropout: Dropout probability on LoRA path
        """
        super().__init__()
        self.base_layer = base_layer
        self.r = r
        self.alpha = alpha
        
        # Freeze base layer
        for param in self.base_layer.parameters():
            param.requires_grad = False
        
        # LoRA matrices
        d, k = base_layer.weight.shape
        self.lora_A = nn.Parameter(torch.randn(r, k) / np.sqrt(r))
        self.lora_B = nn.Parameter(torch.zeros(d, r))
        
        # Scaling factor
        self.scaling = alpha / r
        
        # Optional dropout
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
    
    def forward(self, x):
        # Base model forward pass (frozen)
        base_out = self.base_layer(x)
        
        # LoRA forward pass
        # x: (batch, seq, k)
        # A: (r, k) -> (k, r)
        # B: (d, r)
        lora_out = (self.dropout(x) @ self.lora_A.T) @ self.lora_B.T
        
        return base_out + lora_out * self.scaling
    
    def merge_weights(self):
        """
        Merge LoRA weights into base layer for inference.
        After merging, there's no additional inference cost.
        """
        if hasattr(self, '_merged') and self._merged:
            return
        
        # Compute delta_W = B @ A * scaling
        delta_W = (self.lora_B @ self.lora_A) * self.scaling
        
        # Add to base weights
        self.base_layer.weight.data += delta_W
        self._merged = True
    
    def unmerge_weights(self):
        """
        Separate LoRA weights from base layer.
        Useful for swapping between different LoRA adapters.
        """
        if not (hasattr(self, '_merged') and self._merged):
            return
        
        # Compute delta_W = B @ A * scaling
        delta_W = (self.lora_B @ self.lora_A) * self.scaling
        
        # Subtract from base weights
        self.base_layer.weight.data -= delta_W
        self._merged = False

# Example usage
print("Creating example linear layer with LoRA:")
d, k = 1024, 1024
base = nn.Linear(k, d)
lora_layer = LoRALayer(base, r=16, alpha=32)

# Count parameters
base_params = sum(p.numel() for p in base.parameters())
trainable_params = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in lora_layer.parameters())

print(f"\nBase layer: {d}x{k} = {base_params:,} parameters")
print(f"LoRA trainable: {trainable_params:,} parameters ({trainable_params/base_params*100:.2f}%)")
print(f"Total parameters: {total_params:,}")

# Test forward pass
batch_size, seq_len = 4, 128
x = torch.randn(batch_size, seq_len, k)
output = lora_layer(x)
print(f"\nInput shape: {x.shape}")
print(f"Output shape: {output.shape}")

## The Rank Parameter (r)

### What Does r Control?
The rank $r$ determines:
1. **Capacity**: How much the model can adapt
2. **Parameters**: $\text{params} = (d + k) \times r$
3. **Memory**: Directly proportional to $r$

### Typical Values
From empirical studies (LoRA paper + community):

| Task Type | Recommended r | Notes |
|-----------|--------------|-------|
| Simple classification | 1-4 | Very constrained |
| Text classification | 4-8 | Standard choice |
| Question answering | 8-16 | Moderate complexity |
| Instruction tuning | 16-64 | High capacity needed |
| Domain adaptation | 32-128 | Extensive changes |

### Choosing r: Rules of Thumb
1. Start with $r = 8$ or $r = 16$
2. Increase if:
   - Underfitting (training loss not decreasing)
   - Complex task requiring significant adaptation
   - Large domain shift from pre-training
3. Decrease if:
   - Overfitting (large train-val gap)
   - Simple task or small dataset
   - Memory constraints

### Alpha vs Rank
Common choices:
- $\alpha = r$: Standard scaling
- $\alpha = 2r$: Stronger LoRA influence
- $\alpha = r/2$: More conservative adaptation

Effect: $\alpha$ controls the magnitude of updates without changing capacity

In [ ]:
# Analyze rank vs parameters for different model sizes
model_configs = {
    'GPT-2 Small': {'d': 768, 'layers': 12, 'heads': 12},
    'GPT-2 Medium': {'d': 1024, 'layers': 24, 'heads': 16},
    'GPT-2 Large': {'d': 1280, 'layers': 36, 'heads': 20},
    'GPT-2 XL': {'d': 1600, 'layers': 48, 'heads': 25},
    'Llama 7B': {'d': 4096, 'layers': 32, 'heads': 32},
    'Llama 13B': {'d': 5120, 'layers': 40, 'heads': 40},
}

def compute_lora_params(d, layers, rank, target_modules=4):
    """
    Compute LoRA parameter count.
    
    Args:
        d: Model dimension
        layers: Number of transformer layers
        rank: LoRA rank
        target_modules: Number of matrices per layer to apply LoRA
                       (typically 4: q, k, v, o projections)
    """
    # Each attention matrix: d x d
    # LoRA params per matrix: d*r + r*d = 2*d*r
    params_per_layer = target_modules * 2 * d * rank
    total_params = layers * params_per_layer
    return total_params

# Compute for different ranks
ranks = [1, 2, 4, 8, 16, 32, 64, 128, 256]

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Absolute parameter counts
ax = axes[0, 0]
for model_name, config in model_configs.items():
    params = [compute_lora_params(config['d'], config['layers'], r) / 1e6 
              for r in ranks]
    ax.plot(ranks, params, 'o-', label=model_name, linewidth=2, markersize=6)
ax.set_xlabel('LoRA Rank (r)')
ax.set_ylabel('LoRA Parameters (Millions)')
ax.set_title('LoRA Parameter Count vs Rank')
ax.legend()
ax.grid(True)
ax.set_xscale('log')
ax.set_yscale('log')

# Plot 2: Memory usage (assuming fp16)
ax = axes[0, 1]
for model_name, config in model_configs.items():
    params = [compute_lora_params(config['d'], config['layers'], r) 
              for r in ranks]
    memory_mb = [p * 2 / 1e6 for p in params]  # 2 bytes per param (fp16)
    ax.plot(ranks, memory_mb, 'o-', label=model_name, linewidth=2, markersize=6)
ax.set_xlabel('LoRA Rank (r)')
ax.set_ylabel('Memory (MB)')
ax.set_title('LoRA Memory Usage vs Rank (FP16)')
ax.legend()
ax.grid(True)
ax.set_xscale('log')
ax.set_yscale('log')

# Plot 3: Training memory (params + gradients + optimizer)
ax = axes[1, 0]
# Adam optimizer: params (2 bytes) + gradients (2 bytes) + 
#                 momentum (4 bytes) + variance (4 bytes) = 12 bytes total
for model_name, config in model_configs.items():
    params = [compute_lora_params(config['d'], config['layers'], r) 
              for r in ranks]
    memory_mb = [p * 12 / 1e6 for p in params]
    ax.plot(ranks, memory_mb, 'o-', label=model_name, linewidth=2, markersize=6)
ax.set_xlabel('LoRA Rank (r)')
ax.set_ylabel('Training Memory (MB)')
ax.set_title('LoRA Training Memory vs Rank (FP16 + Adam)')
ax.legend()
ax.grid(True)
ax.set_xscale('log')
ax.set_yscale('log')

# Plot 4: Percentage of full model parameters (Llama 7B example)
ax = axes[1, 1]
llama_7b_total = 7e9  # 7 billion parameters
config = model_configs['Llama 7B']
lora_params = [compute_lora_params(config['d'], config['layers'], r) 
               for r in ranks]
percentages = [lp / llama_7b_total * 100 for lp in lora_params]
ax.bar(range(len(ranks)), percentages, alpha=0.7, color='steelblue')
ax.set_xticks(range(len(ranks)))
ax.set_xticklabels([f'r={r}' for r in ranks], rotation=45)
ax.set_ylabel('% of Full Model Parameters')
ax.set_title('LoRA Parameters as % of Llama 7B Total')
ax.grid(True, axis='y')
ax.set_yscale('log')

# Add value labels on bars
for i, (r, pct) in enumerate(zip(ranks, percentages)):
    ax.text(i, pct, f'{pct:.3f}%', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

# Print table for Llama 7B
print("\nLoRA Parameter Breakdown for Llama 7B:")
print(f"{'Rank':>6} | {'Params':>12} | {'% of Full':>10} | {'Memory (MB)':>12} | {'Train Mem (MB)':>15}")
print("-" * 70)
for r, params, pct in zip(ranks, lora_params, percentages):
    mem_mb = params * 2 / 1e6
    train_mb = params * 12 / 1e6
    print(f"{r:6d} | {params:12,d} | {pct:9.4f}% | {mem_mb:12.1f} | {train_mb:15.1f}")

## Mathematical Derivation

### Forward Pass
Original layer:
$$h = W_0 x$$

With LoRA:
$$h = W_0 x + \frac{\alpha}{r} BAx$$

### Backward Pass (Gradients)
Given loss $L$ and gradient $\frac{\partial L}{\partial h}$:

1. **Gradient w.r.t. B:**
$$\frac{\partial L}{\partial B} = \frac{\alpha}{r} \frac{\partial L}{\partial h} (Ax)^T$$

2. **Gradient w.r.t. A:**
$$\frac{\partial L}{\partial A} = \frac{\alpha}{r} B^T \frac{\partial L}{\partial h} x^T$$

3. **Gradient w.r.t. input:**
$$\frac{\partial L}{\partial x} = W_0^T \frac{\partial L}{\partial h} + \frac{\alpha}{r} A^T B^T \frac{\partial L}{\partial h}$$

Note: Gradients w.r.t. $W_0$ are not computed (frozen)

### Memory Analysis
For a layer with dimensions $d \times k$ and rank $r$:

**Full Fine-tuning:**
- Parameters: $dk$
- Gradients: $dk$
- Adam states: $2dk$ (momentum + variance)
- **Total: $4dk$ (in fp32)**

**LoRA:**
- Parameters: $2r(d + k)$
- Gradients: $2r(d + k)$
- Adam states: $4r(d + k)$
- **Total: $8r(d + k)$ (in fp32)**

**Reduction factor:**
$$\frac{8r(d + k)}{4dk} = \frac{2r(d + k)}{dk}$$

For $d = k = 4096$ and $r = 16$:
$$\frac{2 \times 16 \times 8192}{4096 \times 4096} = 0.0156 = 1.56\%$$

In [ ]:
# Implement gradient computation for LoRA
class LoRAWithGradients:
    """
    Manual implementation of LoRA forward and backward pass.
    For educational purposes - shows exactly how gradients flow.
    """
    def __init__(self, d, k, r, alpha):
        self.d = d
        self.k = k
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r
        
        # Initialize weights
        self.W0 = np.random.randn(d, k) * 0.02  # Frozen
        self.A = np.random.randn(r, k) / np.sqrt(r)
        self.B = np.zeros((d, r))
    
    def forward(self, x):
        """
        Forward pass: h = W0 @ x + scaling * B @ A @ x
        
        Args:
            x: Input (batch, k)
        Returns:
            h: Output (batch, d)
        """
        # Save for backward
        self.x = x
        
        # Base forward
        self.base_out = x @ self.W0.T  # (batch, d)
        
        # LoRA forward
        self.Ax = x @ self.A.T  # (batch, r)
        self.lora_out = self.Ax @ self.B.T  # (batch, d)
        
        h = self.base_out + self.scaling * self.lora_out
        return h
    
    def backward(self, grad_h):
        """
        Backward pass.
        
        Args:
            grad_h: Gradient w.r.t. output (batch, d)
        Returns:
            grad_x: Gradient w.r.t. input (batch, k)
        """
        batch_size = self.x.shape[0]
        
        # Gradient w.r.t. B: (d, r)
        # dL/dB = scaling * dL/dh @ (Ax)^T
        self.grad_B = self.scaling * (grad_h.T @ self.Ax) / batch_size
        
        # Gradient w.r.t. A: (r, k)
        # dL/dA = scaling * B^T @ dL/dh @ x^T
        grad_lora = self.scaling * grad_h @ self.B  # (batch, r)
        self.grad_A = (grad_lora.T @ self.x) / batch_size
        
        # Gradient w.r.t. x: (batch, k)
        grad_x_base = grad_h @ self.W0  # Base model contribution
        grad_x_lora = grad_lora @ self.A  # LoRA contribution
        grad_x = grad_x_base + grad_x_lora
        
        return grad_x

# Test gradient computation
print("Testing gradient computation:")
d, k, r = 512, 512, 8
alpha = 16
batch_size = 32

lora = LoRAWithGradients(d, k, r, alpha)
x = np.random.randn(batch_size, k)

# Forward pass
h = lora.forward(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {h.shape}")

# Backward pass
grad_h = np.random.randn(batch_size, d)
grad_x = lora.backward(grad_h)
print(f"\nGradient shapes:")
print(f"  grad_B: {lora.grad_B.shape}")
print(f"  grad_A: {lora.grad_A.shape}")
print(f"  grad_x: {grad_x.shape}")

# Verify with PyTorch autograd
print("\nVerifying with PyTorch autograd:")
x_torch = torch.tensor(x, dtype=torch.float32, requires_grad=True)
W0_torch = torch.tensor(lora.W0, dtype=torch.float32)
A_torch = torch.tensor(lora.A, dtype=torch.float32, requires_grad=True)
B_torch = torch.tensor(lora.B, dtype=torch.float32, requires_grad=True)
grad_h_torch = torch.tensor(grad_h, dtype=torch.float32)

# Forward
h_torch = x_torch @ W0_torch.T + (alpha / r) * (x_torch @ A_torch.T @ B_torch.T)

# Backward
h_torch.backward(grad_h_torch)

# Compare
print(f"Grad A match: {np.allclose(lora.grad_A, A_torch.grad.numpy(), atol=1e-5)}")
print(f"Grad B match: {np.allclose(lora.grad_B, B_torch.grad.numpy(), atol=1e-5)}")
print(f"Grad x match: {np.allclose(grad_x, x_torch.grad.numpy(), atol=1e-5)}")

## Comparison: Full Fine-tuning vs LoRA

### Parameter Counts: GPT-2 Examples

#### GPT-2 Small (117M parameters)
- Attention matrices per layer: 4 × (768 × 768) = 2.36M
- Total for 12 layers: 28.3M
- **LoRA (r=8)**: 2 × 768 × 8 × 4 × 12 = 589K (2.1%)
- **LoRA (r=16)**: 1.18M (4.2%)

#### GPT-2 Large (774M parameters)
- Attention matrices per layer: 4 × (1280 × 1280) = 6.55M
- Total for 36 layers: 236M
- **LoRA (r=8)**: 2 × 1280 × 8 × 4 × 36 = 2.95M (1.2%)
- **LoRA (r=16)**: 5.90M (2.5%)

#### Llama 2 7B (7B parameters)
- Attention matrices per layer: 4 × (4096 × 4096) = 67.1M
- Total for 32 layers: 2.15B
- **LoRA (r=8)**: 2 × 4096 × 8 × 4 × 32 = 8.39M (0.12%)
- **LoRA (r=16)**: 16.78M (0.24%)
- **LoRA (r=64)**: 67.1M (0.96%)

### Memory Breakdown

For Llama 2 7B with batch_size=1, seq_len=512:

| Component | Full FT | LoRA (r=16) | Savings |
|-----------|---------|-------------|----------|
| Model weights (fp16) | 14 GB | 14 GB | 0 GB |
| Trainable params (fp16) | 14 GB | 33 MB | 13.97 GB |
| Gradients (fp16) | 14 GB | 33 MB | 13.97 GB |
| Optimizer states (fp32) | 56 GB | 134 MB | 55.87 GB |
| Activations (fp16) | 8 GB | 8 GB | 0 GB |
| **Total** | **106 GB** | **22.2 GB** | **83.8 GB** |

**Result**: LoRA reduces training memory by 79%

In [ ]:
# Comprehensive comparison table
def memory_breakdown(model_params, trainable_params, batch_size=1, seq_len=512, d=4096):
    """
    Compute memory breakdown for training.
    
    Args:
        model_params: Total model parameters
        trainable_params: Trainable parameters
        batch_size: Batch size
        seq_len: Sequence length
        d: Model dimension
    
    Returns:
        Dictionary with memory components in GB
    """
    # Model weights (always fp16, always loaded)
    model_weights = model_params * 2 / 1e9  # 2 bytes per param
    
    # Trainable parameters (fp16)
    trainable_mem = trainable_params * 2 / 1e9
    
    # Gradients (fp16)
    gradients = trainable_params * 2 / 1e9
    
    # Optimizer states (fp32)
    # Adam: momentum (4 bytes) + variance (4 bytes) = 8 bytes per param
    optimizer = trainable_params * 8 / 1e9
    
    # Activations (rough estimate, depends on many factors)
    # For transformers: ~2 bytes per token per dimension per layer
    # This is a simplified estimate
    num_layers = 32  # Typical for 7B model
    activations = batch_size * seq_len * d * num_layers * 2 / 1e9
    
    total = model_weights + trainable_mem + gradients + optimizer + activations
    
    return {
        'model_weights': model_weights,
        'trainable_params': trainable_mem,
        'gradients': gradients,
        'optimizer': optimizer,
        'activations': activations,
        'total': total
    }

# Compare different configurations
llama_7b = 7e9
attention_params = 4 * 4096 * 4096 * 32  # 4 matrices, 32 layers

configs = {
    'Full Fine-tuning': {
        'trainable': llama_7b,
        'color': 'red'
    },
    'LoRA (r=4)': {
        'trainable': 2 * 4096 * 4 * 4 * 32,
        'color': 'lightblue'
    },
    'LoRA (r=8)': {
        'trainable': 2 * 4096 * 8 * 4 * 32,
        'color': 'blue'
    },
    'LoRA (r=16)': {
        'trainable': 2 * 4096 * 16 * 4 * 32,
        'color': 'darkblue'
    },
    'LoRA (r=64)': {
        'trainable': 2 * 4096 * 64 * 4 * 32,
        'color': 'navy'
    },
}

# Compute memory for each configuration
results = {}
for name, config in configs.items():
    results[name] = memory_breakdown(llama_7b, config['trainable'])
    results[name]['color'] = config['color']

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Stacked bar chart
components = ['model_weights', 'trainable_params', 'gradients', 'optimizer', 'activations']
labels = ['Model Weights', 'Trainable Params', 'Gradients', 'Optimizer States', 'Activations']
colors_comp = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

x = np.arange(len(configs))
width = 0.6

bottom = np.zeros(len(configs))
for i, (comp, label, color) in enumerate(zip(components, labels, colors_comp)):
    values = [results[name][comp] for name in configs.keys()]
    axes[0].bar(x, values, width, label=label, bottom=bottom, color=color, alpha=0.8)
    bottom += values

axes[0].set_ylabel('Memory (GB)')
axes[0].set_title('Memory Breakdown: Llama 7B Training\n(batch=1, seq_len=512)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(configs.keys(), rotation=45, ha='right')
axes[0].legend(loc='upper left')
axes[0].grid(True, axis='y', alpha=0.3)

# Total memory comparison
total_mems = [results[name]['total'] for name in configs.keys()]
bars = axes[1].bar(x, total_mems, width, 
                    color=[configs[name]['color'] for name in configs.keys()],
                    alpha=0.8)
axes[1].set_ylabel('Total Memory (GB)')
axes[1].set_title('Total Training Memory Comparison')
axes[1].set_xticks(x)
axes[1].set_xticklabels(configs.keys(), rotation=45, ha='right')
axes[1].grid(True, axis='y', alpha=0.3)

# Add value labels on bars
for bar, mem in zip(bars, total_mems):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                f'{mem:.1f} GB',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# Print detailed table
print("\nDetailed Memory Breakdown (Llama 7B):")
print("=" * 100)
header = f"{'Configuration':<20} | {'Trainable':<12} | {'Total Mem':<10} | {'vs Full FT':<12} | {'GPU':<15}"
print(header)
print("=" * 100)

full_ft_mem = results['Full Fine-tuning']['total']
for name in configs.keys():
    res = results[name]
    trainable_m = res['trainable_params'] * 1000  # Convert to MB
    total = res['total']
    saving = (1 - total/full_ft_mem) * 100
    
    # Determine GPU compatibility
    if total <= 16:
        gpu = "RTX 4080 (16GB)"
    elif total <= 24:
        gpu = "RTX 3090 (24GB)"
    elif total <= 40:
        gpu = "A100 (40GB)"
    elif total <= 80:
        gpu = "A100 (80GB)"
    else:
        gpu = "Multi-GPU needed"
    
    print(f"{name:<20} | {trainable_m:>10.1f} MB | {total:>8.1f} GB | {saving:>9.1f}% | {gpu:<15}")

print("=" * 100)

## Summary: Why LoRA Works

### Key Insights
1. **Intrinsic Dimensionality**: Fine-tuning updates live in a low-dimensional subspace
2. **Matrix Decomposition**: Low-rank matrices can approximate these updates
3. **Frozen Base Model**: Keep pre-trained knowledge intact, only train small adapters
4. **Zero Inference Overhead**: Merge weights after training

### Advantages
- **Memory**: 10-100x reduction in trainable parameters
- **Speed**: Faster training (fewer gradient computations)
- **Quality**: Matches full fine-tuning performance on most tasks
- **Deployment**: Store one base model + multiple small adapters
- **Flexibility**: Easy to swap between tasks at inference

### Limitations
- **Not universal**: Some tasks need more capacity (r > 128)
- **Architecture dependent**: Need to choose target modules carefully
- **Hyperparameters**: r and alpha require tuning
- **Catastrophic forgetting**: Can still occur if r too small

### When to Use LoRA
**Use LoRA when:**
- Limited GPU memory
- Need to fine-tune large models (>1B params)
- Want to deploy multiple task-specific models
- Task is reasonably close to pre-training distribution

**Use full fine-tuning when:**
- Extreme domain shift (e.g., code → biology)
- Small models (<500M params) where memory isn't an issue
- Absolute best performance needed
- Have sufficient computational resources

### Historical Impact
LoRA (2021) democratized large model fine-tuning:
- Before: Only large labs with 8×A100 clusters could fine-tune GPT-3 scale
- After: Individual researchers with single consumer GPU can fine-tune 7B models
- Led to explosion of specialized models and tools (Alpaca, Vicuna, etc.)
- Foundation for QLoRA (2023) → fine-tune 65B on single GPU

### Next Steps
In the following notebooks:
- **21_lora_llama2_7b.ipynb**: Practical LoRA implementation
- **22_qlora_implementation.ipynb**: Combining LoRA with quantization
- **23_lora_target_modules.ipynb**: Advanced target module selection
- **24_custom_loss_with_lora.ipynb**: Custom training objectives

In [ ]:
# Final visualization: Timeline of PEFT methods
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 8))

# Timeline data
events = [
    ('2017-06', 'Transformer', 'Foundation', 0.5),
    ('2018-10', 'BERT', 'Foundation', 0.5),
    ('2019-03', 'Adapter Layers', 'PEFT', 1.5),
    ('2020-05', 'GPT-3', 'Foundation', 0.5),
    ('2021-04', 'Prompt Tuning', 'PEFT', 1.5),
    ('2021-06', 'LoRA', 'PEFT', 2.5),
    ('2021-10', 'BitFit', 'PEFT', 1.5),
    ('2022-11', 'ChatGPT', 'Foundation', 0.5),
    ('2023-02', 'Llama', 'Foundation', 0.5),
    ('2023-05', 'QLoRA', 'PEFT', 2.5),
    ('2023-07', 'Llama 2', 'Foundation', 0.5),
]

# Convert dates
from datetime import datetime
dates = [datetime.strptime(date, '%Y-%m') for date, _, _, _ in events]
names = [name for _, name, _, _ in events]
types = [type_ for _, _, type_, _ in events]
importance = [imp for _, _, _, imp in events]

# Plot
colors = {'Foundation': 'steelblue', 'PEFT': 'coral'}
y_pos = {'Foundation': 1, 'PEFT': 2}

for date, name, type_, imp in zip(dates, names, types, importance):
    y = y_pos[type_]
    ax.scatter(date, y, s=imp*100, c=colors[type_], alpha=0.7, edgecolors='black', linewidths=2)
    ax.text(date, y + 0.15, name, ha='center', va='bottom', fontsize=9, fontweight='bold')

# Highlight LoRA
lora_idx = names.index('LoRA')
lora_date = dates[lora_idx]
ax.scatter(lora_date, 2, s=500, c='red', marker='*', zorder=10, 
          edgecolors='darkred', linewidths=3, label='LoRA (this notebook)')

ax.set_ylim(0.5, 2.5)
ax.set_yticks([1, 2])
ax.set_yticklabels(['Foundation Models', 'PEFT Methods'])
ax.set_xlabel('Year')
ax.set_title('Evolution of Parameter-Efficient Fine-Tuning\n' + 
            'LoRA: The Breakthrough Moment (June 2021)', fontsize=14, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
ax.legend(loc='upper left')

# Add annotations
ax.annotate('GPT-3: Too large to fine-tune\n(175B params, 2.5TB memory)', 
           xy=(dates[3], 1), xytext=(dates[3], 0.6),
           arrowprops=dict(arrowstyle='->', color='red', lw=2),
           fontsize=9, ha='center')

ax.annotate('LoRA: Fine-tune with 0.01% params\nSame quality, 100x less memory', 
           xy=(lora_date, 2), xytext=(lora_date, 2.4),
           arrowprops=dict(arrowstyle='->', color='green', lw=2),
           fontsize=9, ha='center', fontweight='bold',
           bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

plt.tight_layout()
plt.show()

print("\nLoRA Impact:")
print("- Made fine-tuning accessible to individual researchers")
print("- Enabled multi-task model deployment (one base + many adapters)")
print("- Foundation for QLoRA and modern PEFT methods")
print("- Used in production: Stable Diffusion, LLaMA fine-tuning, and more")